In [1]:
import json

with open("model/metadata.json", encoding="utf-8") as f:
    metadata = json.load(f)

print(metadata)

{'threshold': 0.6828784680890222, 'classes': ['ham', 'spam'], 'positive_label': 'spam', 'dataset': 'jason1966/aiwithcagri_trke-spam-sms-ve-sms-kategorileri', 'split': {'train': 8580, 'validation': 2860, 'test': 2861}, 'class_distribution': {'ham': 11672, 'spam': 2629}, 'random_state': 42, 'sklearn_version': '1.9.1', 'python_version': '3.11.17', 'trained_at': '2026-10-17T14:10:27.082342+00:00'}


In [1]:
import pandas as pd

df = pd.read_csv("data/turkish_spam_combined.csv")

mask = df["message"].str.contains(
    "bahis",
    case=False,
    regex=False,
    na=False
)

print(pd.crosstab(
    df.loc[mask, "label"],
    columns="Mesaj sayısı"
))

col_0  Mesaj sayısı
label              
ham               8
spam            619


In [2]:
normal_bahis = df[
    (df["label"] == "ham") &
    (df["message"].str.contains(
        "bahis",
        case=False,
        regex=False,
        na=False
    ))
]

for i, message in enumerate(normal_bahis["message"], 1):
    print(f"{i}. {message}\n")

1. Yasadışı bahis sitelerine erişim engellenmiştir. Güvenliğiniz için resmi kurumların duyurularını takip ediniz.

2. Bugünün spor haberleri ve maç analizleri sitemizde! En güncel bilgiler ve bahis tahminleri için bizi takip edin.

3. Yasa disi sitelerden bahis oynamak ceza gerektirir. Lütfen dikkatli olunuz, yasal siteleri tercih ediniz.

4. Yasal olmayan bahis sitelerinden gelen davetleri engelleyin. Guvenliğiniz için sadece resmi kurumları tercih edin.

5. Bahis kuponunuz tuttu! Kazancınız hesabınıza başarıyla yatırılmıştır.

6. Yasa dışı bahis sitelerine güvenmeyin. Paranızı riske atmayın, yasal platformlarda kalın.

7. Bahis oynamak için yaş sınırı 18'dir. Sorumlu oyun kurallarına uyun.

8. Bahis oynamak için yaş sınırı olduğunu hatırlatırız. Lütfen kurallara uyun.



In [3]:
mask = df["message"].str.contains(
    "bedava",
    case=False,
    regex=False,
    na=False
)

print(df.loc[mask, "label"].value_counts())

label
spam    209
ham      20
Name: count, dtype: int64


In [8]:
import train_bilstm
from tensorflow import keras

model = keras.models.load_model(
    "model/spam_classifier_v3.keras",
    custom_objects={
        "spam_v3>turkish_standardize": train_bilstm.turkish_standardize
    },
    compile=False
)

vectorizer = model.get_layer("text_vectorization")

for text in ["O*ospu", "Or*spu", "Y*vşak", "Yavşak", "porno"]:
    print(text, "→", vectorizer([text]).numpy()[0][:10])

O*ospu → [284   1   0   0   0   0   0   0   0   0]
Or*spu → [12697     1     0     0     0     0     0     0     0     0]
Y*vşak → [3395    1    0    0    0    0    0    0    0    0]
Yavşak → [1 0 0 0 0 0 0 0 0 0]
porno → [1 0 0 0 0 0 0 0 0 0]
